# Notebook 03: Foundation Model Training

We train a **Llama-architecture decoder model** on the tokenized transaction sequences from notebook 02 using **causal language modeling** (next-token prediction). At each position, the model predicts the next token from all preceding tokens, learning transaction patterns without any fraud labels.

```
Input:  <bos>  AMT_3  MERCH_1498  CAT_RETAIL  MCC_5912  HOUR_05  DOW_0  ...
Target:        AMT_3  MERCH_1498  CAT_RETAIL  MCC_5912  HOUR_05  DOW_0  MONTH_05  ...
```

Every token position provides a training signal, so through millions of predictions the model builds rich representations of spending patterns, merchant relationships, and temporal behaviors.

## Model Configuration

| Parameter | Value | Notes |
|-----------|-------|-------|
| **Architecture** | Llama (Decoder-only Transformer) | Causal (autoregressive) attention |
| **Hidden Size** | 512 | Transformer hidden dimension |
| **Layers** | 8 | Transformer depth |
| **Attention Heads** | 8 query / 2 KV | Grouped-Query Attention (GQA) |
| **FFN Size** | 1408 | SwiGLU activation |
| **Context Window** | 8192 | RoPE positional encoding |
| **Parameters** | **≈29M** | Compact embedding table |
| **Vocab Size** | ≈6,252 | Matches `FinancialTokenizerPipeline` (merchant_hash_size=2,000) |
| **Training Data** | ≈263M tokens | ≈64K sequences from 19.5M transactions |

## Training Pipeline

Training uses `scripts/train_decoder_model.py` with **NeMo AutoModel** and FSDP2 for distributed training. Configuration is defined in `configs/pretrain_financial_llama.yaml`; custom tokenizer and dataset are integrated via the `_target_` syntax. Scaling from 1 GPU to multi-node requires only a `torchrun` wrapper -- no code changes.

## Custom Components

We only provide two domain-specific components -- everything else (FSDP2, optimizer, checkpointing, mixed precision) is handled by NeMo AutoModel:

1. **`FinancialTabularTokenizer`** -- custom tokenization for transaction data
2. **`FinancialCLMDataset`** -- next-token prediction data loading (referenced in YAML via `_target_`)

Checkpoints are saved in HuggingFace format (`save_consolidated: true`), enabling downstream use with `AutoModelForCausalLM.from_pretrained()`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "decoder_corpus"
CORPUS_PATH = CORPUS_DIR / "train_corpus.txt"
VAL_CORPUS_PATH = CORPUS_DIR / "val_corpus.txt"

YAML_CONFIG = PROJECT_ROOT / "configs/pretrain_financial_llama.yaml"
TRAIN_SCRIPT = PROJECT_ROOT / "scripts/train_decoder_model.py"

PRETRAINED_MODEL = PROJECT_ROOT / "models/decoder-foundation-model"

# Verify prerequisites from notebook 02
assert CORPUS_PATH.exists(), f"Training corpus not found: {CORPUS_PATH}  (run notebook 02 first)"
assert VAL_CORPUS_PATH.exists(), f"Validation corpus not found: {VAL_CORPUS_PATH}  (run notebook 02 first)"
assert YAML_CONFIG.exists(), f"YAML config not found: {YAML_CONFIG}"
assert TRAIN_SCRIPT.exists(), f"Training script not found: {TRAIN_SCRIPT}"

# Pre-trained checkpoint (used by notebook 04 for inference)
if PRETRAINED_MODEL.exists() and (PRETRAINED_MODEL / "config.json").exists():
    print(f"Pre-trained decoder checkpoint available for notebook 04:")
    print(f"  {PRETRAINED_MODEL}\n")

print(f"Training corpus : {CORPUS_PATH}")
print(f"Validation      : {VAL_CORPUS_PATH} ({'exists' if VAL_CORPUS_PATH.exists() else 'NOT FOUND'})")
print(f"YAML config     : {YAML_CONFIG}")
print(f"Pre-trained     : {PRETRAINED_MODEL}")

Training corpus : /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/data/decoder_corpus/train_corpus.txt
Validation      : /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/data/decoder_corpus/val_corpus.txt (exists)
YAML config     : /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/configs/pretrain_financial_llama.yaml
Pre-trained     : /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/models/decoder-foundation-model


## 1. Launch Training

Training is configured through `configs/pretrain_financial_llama.yaml` (model architecture, AdamW + cosine annealing optimizer, FSDP2 distribution, consolidated HF checkpoints). The cell below runs a **30-step demo** on 1 GPU (≈2 min). The pre-trained model used in notebook 04 was trained for ≈3,000 steps on 8x A100.

The random baseline loss is `ln(≈6,252) ≈ 8.7`; loss should drop to ≈5–6 within 30 steps.

### YAML Configuration

In [2]:
print(f"=== {YAML_CONFIG} ===\n")
with open(YAML_CONFIG) as f:
    print(f.read())

=== /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/configs/pretrain_financial_llama.yaml ===

# Financial Foundation Model Pretraining — Decoder-Only (Llama Architecture)
#
# NeMo AutoModel configuration for pretraining a small Llama on
# financial transaction sequences using causal language modeling.
#
# Key architectural choices (vs GPT-2):
#   - RoPE (Rotary Position Embeddings) instead of learned absolute
#   - GQA (Grouped Query Attention): 8 query heads, 2 KV heads
#   - SwiGLU activation instead of GELU
#   - RMSNorm instead of LayerNorm
#   - No dropout (standard for Llama-family architectures)
#   - 8192 max context (RoPE allows extrapolation beyond training length)
#
# Parameter count: ~29M
#
# Launch methods:
#
#   1. Multi-GPU (recommended):
#      torchrun --nproc-per-node=8 scripts/train_decoder_model.py \
#          -c configs/pretrain_financial_llama.yaml \
#          --dataset.data_path data/decoder_corpus/train_corpus.txt \
#          --validation_da

In [3]:
import subprocess

# Fresh training demo: 30 steps from scratch on 1 GPU.
# The loss will drop rapidly from ~8.7 (random baseline for ~6K vocab) demonstrating
# that the model is learning transaction patterns. The full pre-trained checkpoint
# is available for notebook 04.
DEMO_CHECKPOINT_DIR = PROJECT_ROOT / "models" / "decoder-demo" / "checkpoints"
DEMO_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

PYTHON = "/opt/venv/bin/python"

cmd = [
    PYTHON, str(TRAIN_SCRIPT),
    "-c", str(YAML_CONFIG),
    "--dataset.data_path", str(CORPUS_PATH),
    "--validation_dataset.data_path", str(VAL_CORPUS_PATH),
    "--step_scheduler.max_steps", "30",
    "--step_scheduler.global_batch_size", "16",
    "--step_scheduler.local_batch_size", "16",
    "--lr_scheduler.lr_warmup_steps", "10",
    "--step_scheduler.val_every_steps", "15",
    "--step_scheduler.ckpt_every_steps", "15",
    "--checkpoint.checkpoint_dir", str(DEMO_CHECKPOINT_DIR),
]

print("Launching training (30 steps, 1 GPU):")
print(" ".join(cmd))
print()

result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode != 0:
    print(f"\nTraining exited with code {result.returncode}")
else:
    print("\nTraining complete!")
    print(f"\nThe full pre-trained model (trained on 8x A100) is available at:")
    print(f"  {PRETRAINED_MODEL}")

# Show the torchrun command for full distributed training
print("\n" + "=" * 70)
print("For full distributed training (8 GPUs):")
print("=" * 70)
print(f"\n  torchrun --nproc-per-node=8 {TRAIN_SCRIPT} \\")
print(f"      -c {YAML_CONFIG} \\")
print(f"      --dataset.data_path {CORPUS_PATH} \\")
print(f"      --validation_dataset.data_path {VAL_CORPUS_PATH}")

Launching training (30 steps, 1 GPU):
/opt/venv/bin/python /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/scripts/train_decoder_model.py -c /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/configs/pretrain_financial_llama.yaml --dataset.data_path /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/data/decoder_corpus/train_corpus.txt --validation_dataset.data_path /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/data/decoder_corpus/val_corpus.txt --step_scheduler.max_steps 30 --step_scheduler.global_batch_size 16 --step_scheduler.local_batch_size 16 --lr_scheduler.lr_warmup_steps 10 --step_scheduler.val_every_steps 15 --step_scheduler.ckpt_every_steps 15 --checkpoint.checkpoint_dir /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/models/decoder-demo/checkpoints



/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Skipping import of cpp extensions due to incompatible torch version 2.8.0a0+5228986c39.nv25.06 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/opt/venv/lib/python3.12/site-packages/liger_kernel/transformers/multi_token_attention.py:12: SyntaxWarning: invalid escape sequence '\i'
  """
/opt/venv/lib/python3.12/site-packages/pydantic/

cfg-path: /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/configs/pretrain_financial_llama.yaml

Financial Foundation Model — Decoder-Only Pretraining
  Architecture: LlamaConfig
  Hidden size:  512
  Layers:       8
  Vocab size:   6251
  Max steps:    30
  Batch size:   16 global, 16 local

> initializing torch distributed with 1 workers.


2026-03-04 04:16:14 | INFO | nemo_automodel.components.loggers.log_utils | Setting logging level to 20
2026-03-04 04:16:14 | INFO | root | Experiment_details:
2026-03-04 04:16:14 | INFO | root | Timestamp: '2026-03-04T04:16:14'
2026-03-04 04:16:14 | INFO | root | User: root
2026-03-04 04:16:14 | INFO | root | Host: 6b35488265bf
2026-03-04 04:16:14 | INFO | root | World size: 1
2026-03-04 04:16:14 | INFO | root | Backend: nccl
2026-03-04 04:16:14 | INFO | root | Recipe: TrainFinetuneRecipeForNextTokenPrediction
2026-03-04 04:16:14 | INFO | root | Model name: null
2026-03-04 04:16:14 | INFO | root | Recipe config:
2026-03-04 04:16:14 | INFO | root |   model:
2026-03-04 04:16:14 | INFO | root |     _target_: <bound method _BaseNeMoAutoModelClass.from_config of <class 'nemo_automodel.components._transformers.auto_model.NeMoAutoModelForCausalLM'>>
2026-03-04 04:16:14 | INFO | root |     config:
2026-03-04 04:16:14 | INFO | root |       _target_: <class 'transformers.models.llama.configurati

Loading corpus from /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/data/decoder_corpus/train_corpus.txt...
  Loaded 64,335 sequences (seq_length=4096)
Loading corpus from /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/data/decoder_corpus/val_corpus.txt...
  Loaded 9,739 sequences (seq_length=4096)


...

Saving checkpoint to /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/models/decoder-demo/checkpoints/epoch_0_step_14


2026-03-04 04:17:03 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Consolidating safetensors files from /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/models/decoder-demo/checkpoints/epoch_0_step_14/model to /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/models/decoder-demo/checkpoints/epoch_0_step_14/model/consolidated. Beginning at time 1772597823.776043
2026-03-04 04:17:04 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Done consolidating. Took 0.86 secs.
2026-03-04 04:17:43 | INFO | root | [val] step 14 | epoch 0 | loss 6.4048 | lr 1.72e-04 | num_label_tokens 39890944
2026-03-04 04:17:43 | INFO | root | step 15 | epoch 0 | loss 6.7333 | grad_norm 2.9219 | lr 1.60e-04 | mem 13.05 GiB | tps 1607.87(1607.87/gpu) | num_label_tokens 65536
2026-03-04 04:17:43 | INFO | root | step 16 | epoch 0 | loss 6.5260 | grad_norm 2.6562 | lr 1.47e-04 | mem 13.05 GiB | tps 395099.25(395099

Saving checkpoint to /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/models/decoder-demo/checkpoints/epoch_0_step_29


2026-03-04 04:17:46 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Consolidating safetensors files from /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/models/decoder-demo/checkpoints/epoch_0_step_29/model to /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/models/decoder-demo/checkpoints/epoch_0_step_29/model/consolidated. Beginning at time 1772597866.792201
2026-03-04 04:17:47 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Done consolidating. Took 0.85 secs.
2026-03-04 04:18:27 | INFO | root | [val] step 29 | epoch 0 | loss 5.8605 | lr 6.50e-06 | num_label_tokens 39890944



Training complete!

The full pre-trained model (trained on 8x A100) is available at:
  /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/models/decoder-foundation-model

For full distributed training (8 GPUs):

  torchrun --nproc-per-node=8 /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/scripts/train_decoder_model.py \
      -c /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/configs/pretrain_financial_llama.yaml \
      --dataset.data_path /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/data/decoder_corpus/train_corpus.txt \
      --validation_dataset.data_path /workspace/nim-blueprints/nvidia/transaction-foundation-model/ux/data/decoder_corpus/val_corpus.txt


## Checkpoints

With `save_consolidated: true`, checkpoints are in HuggingFace format (`config.json` + `safetensors`), loadable via `AutoModelForCausalLM.from_pretrained()`. The pre-trained checkpoint at `models/decoder-foundation-model/` is ready for inference in notebook 04.

## 2. Monitor Training with TensorBoard

Key curves: `train_loss` (steady decrease), `val_loss` (should track train_loss; divergence signals overfitting), `learning_rate` (warmup then cosine decay).

In [4]:
%load_ext tensorboard
%tensorboard --logdir {PROJECT_ROOT}/models/decoder-demo

## Summary

We trained a ≈29M parameter Llama decoder on ≈64K tokenized transaction sequences using NeMo AutoModel. The model learns to predict the next transaction token from context through self-supervised learning, without any fraud labels. The full pre-trained checkpoint (≈3,000 steps on 8x A100) is available for the next step.

Continue to [04_inference_embedding_extraction.ipynb](./04_inference_embedding_extraction.ipynb).